# Lab 5.4 — Introduction to RAG
**Module 5: Generative AI APIs & Intro to RAG**

In this lab you will:
- **Demonstrate the hallucination problem** concretely — see an LLM confidently invent wrong answers
- Understand the **RAG architecture**: two phases (ingestion and query) and three components (chunker, retriever, generator)
- Build each RAG component **from scratch** using Sentence Transformers + numpy + Gemini
- Measure the **improvement RAG provides** over a direct LLM call

> **Instructor Note:** The hallucination demo in Section 1 is the most important cell in this lab. Students need to *see* the LLM failing before they appreciate why RAG matters. Run it live and ask the class to spot which facts are wrong.

## 📦 Requirements & Troubleshooting

### Required Packages

| Package | Install Name |
|---------|-------------|
| google-generativeai | `google-generativeai` |
| sentence-transformers | `sentence-transformers` |
| numpy | `numpy` |
| pandas | `pandas` |

```bash
pip install google-generativeai sentence-transformers numpy pandas
```

### 🔑 API Key Required
This lab calls the **Google Gemini API** for the generation step.

```bash
export GEMINI_API_KEY="AIza..."
```

Get a free key at: [aistudio.google.com](https://aistudio.google.com/app/apikey)

> The ingestion and retrieval steps (Sections 3–4) run locally with no API key.

In [1]:
import subprocess, sys

required = {
    'google.generativeai': 'google-generativeai',
    'sentence_transformers': 'sentence-transformers',
    'numpy':   'numpy',
    'pandas':  'pandas',
}
for pkg, inst in required.items():
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', inst,
                               '--quiet', '--break-system-packages'])

print('All packages ready ✅')

/var/folders/nr/zr3cbb4j2dz42fkzm9zw80080000gp/T/ipykernel_20013/45683193.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  __import__(pkg)


All packages ready ✅


## 1. The Hallucination Problem

LLMs are trained on general text from the internet. They have no access to your private knowledge base — your internal KB articles, runbooks, architecture docs, or recent incidents. When asked about domain-specific facts they don't know, they often **generate plausible-sounding but incorrect answers** rather than admitting uncertainty.

This is called **hallucination**, and it is the core problem RAG solves.

> **Instructor Note:** The cell below asks Gemini about specific Nutanix implementation details. Some answers will be confidently wrong or vague. Point this out live — it makes the RAG improvement in Section 6 much more impactful.

In [3]:
import os
import google.generativeai as genai

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise EnvironmentError(
        'Set GEMINI_API_KEY before running.\n'
        'Uncomment: os.environ["GEMINI_API_KEY"] = "AIza..."'
    )

genai.configure(api_key=GEMINI_API_KEY)
MODEL = 'gemini-3.1-flash-lite'

def ask_direct(question: str, max_tokens: int = 300) -> str:
    """Ask Gemini directly — no context provided."""
    m = genai.GenerativeModel(MODEL)
    resp = m.generate_content(
        question,
        generation_config=genai.GenerationConfig(max_output_tokens=max_tokens, temperature=0.1)
    )
    return resp.text

# ── Questions the LLM may hallucinate on ─────────────────────────────────
hallucination_tests = [
    'What exact CLI command clears Stargate WAL corruption in Nutanix AOS 6.5?',
    'What is the recommended Cassandra heap size for a Nutanix cluster with 100+ VMs?',
    'Where exactly are the Prism Central API logs stored on the filesystem?',
    'What LACP timer mode does Nutanix AOS use by default?',
    'What is the minimum cluster node count required to enable erasure coding in Nutanix?',
]

print('=== Direct LLM Answers (no context) ===')
print('Watch for vague, invented, or confidently wrong answers.\n')

for q in hallucination_tests:
    answer = ask_direct(q)
    print(f'Q: {q}')
    print(f'A: {answer.strip()[:250]}')
    print()

=== Direct LLM Answers (no context) ===
Watch for vague, invented, or confidently wrong answers.

Q: What exact CLI command clears Stargate WAL corruption in Nutanix AOS 6.5?
A: In Nutanix AOS 6.5, there is **no single "magic" CLI command** to clear Stargate WAL (Write-Ahead Log) corruption. Because WAL corruption often indicates underlying metadata inconsistency or disk-level issues, Nutanix engineering requires a specific 

Q: What is the recommended Cassandra heap size for a Nutanix cluster with 100+ VMs?
A: When sizing Cassandra (the distributed metadata store for Nutanix AOS) for a cluster with 100+ VMs, it is important to understand that **Nutanix automatically manages the Cassandra heap size** based on the amount of RAM assigned to the Controller VM 

Q: Where exactly are the Prism Central API logs stored on the filesystem?
A: In Nutanix Prism Central, the API logs are primarily handled by the **`prism`** service. Because Prism Central is built on a microservices architecture, t

## 2. RAG Architecture

RAG (Retrieval-Augmented Generation) solves hallucination by grounding the LLM in a private knowledge base. It has two phases:

```
INGESTION (offline, runs once)
─────────────────────────────────────────────────────────
  Raw documents
       │
       ▼
  [Chunker]  ──► splits text into overlapping passages (~150 words)
       │
       ▼
  [Embedder] ──► encodes each chunk as a 384-dim vector (Sentence Transformers)
       │
       ▼
  [Index]    ──► stores (chunk_text, embedding, source) in memory / vector DB


QUERY (online, runs per user question)
─────────────────────────────────────────────────────────
  User question
       │
       ▼
  [Embedder]  ──► encodes question as a 384-dim vector
       │
       ▼
  [Retriever] ──► cosine similarity → top-k matching chunks
       │
       ▼
  [Generator] ──► LLM prompt = system + retrieved chunks + question
       │
       ▼
  Grounded answer (with citations)
```

**Why chunking?** LLMs have input token limits. Chunking splits long documents into passages that fit in the prompt. Overlap (e.g. 30 words) ensures no fact is cut across a chunk boundary.

**Why embeddings?** Keyword search misses synonyms and paraphrases. Embeddings match by *meaning* — "IO error" matches "disk failure" even with no shared words.

> **Instructor Note:** Draw this on the whiteboard before running the cells. Students who understand the two-phase architecture can debug RAG failures (retrieval miss vs. generation failure) much more effectively.

## 3. Build the Ingestion Pipeline

We'll ingest 4 short Nutanix KB documents and build an in-memory index. The same pattern scales to thousands of documents with a real vector database (Lab 5.5 production section covers FAISS and ChromaDB).

In [4]:
# ── Sample Nutanix KB documents ─────────────────────────────────────────

DOCUMENTS = [
    {
        'source': 'KB-2001',
        'title':  'Stargate Service Crash-Loop Recovery',
        'text': (
            "Stargate is the Nutanix I/O manager responsible for all read and write operations "
            "in the Nutanix Distributed Storage Fabric. When Stargate enters a crash loop, "
            "follow these diagnostic steps. First, check status with: genesis status | grep -i stargate. "
            "View crash logs at /home/nutanix/data/logs/stargate.FATAL. "
            "Common causes include disk SMART failures (run: ncc health_checks hardware_checks disk_checks all), "
            "WAL corruption (check /home/nutanix/data/stargate-storage/wal/), "
            "and CVM memory pressure (minimum 24 GB RAM required for AOS 6.x). "
            "To clear WAL corruption: stop Stargate with genesis stop_tasks, "
            "rename the WAL directory to wal.bak, then restart with genesis start. "
            "For disk failures, mark the disk bad in Prism Central under Storage > Disk, "
            "then replace the physical drive. Always run NCC after recovery: "
            "ncc health_checks run_all."
        ),
    },
    {
        'source': 'KB-2002',
        'title':  'Cerebro Replication Lag Recovery',
        'text': (
            "Cerebro is the Nutanix data protection service managing async replication, "
            "NearSync (20-second RPO), and Metro Availability (0 RPO). "
            "Replication lag is most commonly caused by WAN bandwidth saturation, "
            "remote site cluster degradation, or NTP clock skew exceeding 5 minutes between sites "
            "(which halts replication entirely). "
            "Diagnose with: ncli pd list-replication-status and cerebro_cli get_replication_stats. "
            "Check remote site connectivity: ping <remote-cvm-ip>. "
            "To check NTP sync: ntpq -p on both sites — offset should be under 5 minutes. "
            "Fix bandwidth issues by enabling Cerebro compression via Prism Protection Domains settings. "
            "Restart Cerebro on the remote site if the service is degraded: "
            "ssh to remote CVM, then genesis stop_tasks && genesis start. "
            "View Cerebro logs at /home/nutanix/data/logs/cerebro.INFO."
        ),
    },
    {
        'source': 'KB-2006',
        'title':  'Network MTU Mismatch Diagnosis',
        'text': (
            "MTU mismatch between CVMs, OVS bridges, and physical switches causes "
            "silent packet drops and storage performance degradation. "
            "Nutanix recommends MTU 9000 (jumbo frames) on storage and backplane networks. "
            "Symptoms: intermittent Stargate latency spikes, large file transfers failing, "
            "NCC network checks failing. "
            "Diagnose with: ip link show | grep mtu on each CVM. "
            "Test path MTU: ping -M do -s 8972 <remote-cvm-ip> (8972 = 9000 minus 28-byte headers). "
            "Check OVS bridge MTU: ovs-vsctl list interface | grep mtu. "
            "Fix CVM NIC MTU: ip link set <nic> mtu 9000 — make persistent via /etc/sysconfig/network-scripts. "
            "Fix OVS bridge: ovs-vsctl set interface <bridge> mtu_request=9000. "
            "For LACP flapping, check LACP timer mode — Nutanix uses fast mode (1-second timers); "
            "ensure switch ports are set to LACP fast as well."
        ),
    },
    {
        'source': 'KB-2007',
        'title':  'NCC Health Check Reference',
        'text': (
            "NCC (Nutanix Cluster Check) is the automated health check framework. "
            "Run a full check with: ncc health_checks run_all. "
            "Run category checks: ncc health_checks hardware_checks all. "
            "Run a single check: ncc health_checks hardware_checks disk_checks disk_smart_check. "
            "Result levels: PASS (no issue), INFO (monitor), WARN (remediate within 1 week), "
            "ERR (remediate immediately — risk of data loss), FAIL (check itself failed). "
            "Key check categories: hardware_checks (disk SMART, memory ECC, NIC), "
            "cvm_checks (memory, services, NTP sync), network_checks (MTU, LACP, DNS), "
            "stargate_checks (IO path, WAL health), cerebro_checks (replication, RPO compliance). "
            "Schedule weekly automated checks by adding to crontab: "
            "0 2 * * 0 /home/nutanix/ncc/bin/ncc health_checks run_all. "
            "Escalate to Nutanix Support for any ERR result not resolved by documented fix."
        ),
    },
]

print(f'Loaded {len(DOCUMENTS)} KB documents')
for doc in DOCUMENTS:
    word_count = len(doc["text"].split())
    print(f'  {doc["source"]}: {doc["title"]} ({word_count} words)')

Loaded 4 KB documents
  KB-2001: Stargate Service Crash-Loop Recovery (119 words)
  KB-2002: Cerebro Replication Lag Recovery (113 words)
  KB-2006: Network MTU Mismatch Diagnosis (124 words)
  KB-2007: NCC Health Check Reference (114 words)


In [5]:
# ── Chunker: sliding window with word overlap ────────────────────────────

def chunk_document(doc: dict, chunk_size: int = 120, overlap: int = 25) -> list:
    """Split a document into overlapping word-window chunks."""
    words = doc['text'].split()
    step  = max(1, chunk_size - overlap)
    chunks = []
    for i in range(0, len(words), step):
        window = words[i : i + chunk_size]
        if len(window) < 15:   # skip tiny trailing fragments
            break
        chunks.append({
            'chunk_id':   len(chunks),
            'source':     doc['source'],
            'title':      doc['title'],
            'text':       ' '.join(window),
            'word_start': i,
        })
    return chunks

# ── Chunk all documents ────────────────────────────────────────────────────
all_chunks = []
for doc in DOCUMENTS:
    doc_chunks = chunk_document(doc)
    all_chunks.extend(doc_chunks)
    print(f'{doc["source"]}: {len(doc["text"].split())} words → {len(doc_chunks)} chunks')

print(f'\nTotal chunks: {len(all_chunks)}')
print(f'\nSample chunk from KB-2001:')
print(f'  "{all_chunks[0]["text"][:120]}..."')
print(f'  word_start={all_chunks[0]["word_start"]}, len={len(all_chunks[0]["text"].split())} words')

KB-2001: 119 words → 2 chunks
KB-2002: 113 words → 2 chunks
KB-2006: 124 words → 2 chunks
KB-2007: 114 words → 2 chunks

Total chunks: 8

Sample chunk from KB-2001:
  "Stargate is the Nutanix I/O manager responsible for all read and write operations in the Nutanix Distributed Storage Fab..."
  word_start=0, len=119 words


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

print('Loading embedding model...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# ── Embed all chunks ──────────────────────────────────────────────────────
chunk_texts = [c['text'] for c in all_chunks]

t0 = time.perf_counter()
chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=False)
elapsed = (time.perf_counter() - t0) * 1000

print(f'Embedded {len(all_chunks)} chunks in {elapsed:.0f}ms')
print(f'Index shape: {chunk_embeddings.shape}  '
      f'({chunk_embeddings.shape[0]} chunks × {chunk_embeddings.shape[1]} dims)')
print(f'Index memory: {chunk_embeddings.nbytes / 1024:.1f} KB')

Loading embedding model...


I0529 13:31:01.608698  266232 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(90, generation: 1)
I0529 13:31:01.608730  266232 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0529 13:31:01.608733  266232 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(90, generation: 1)
I0529 13:31:01.608734  266232 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0529 13:31:01.608736  266232 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(90, generation: 1)
I0529 13:31:01.608773  266232 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0529 13:31:01.608783  266232 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(90, generation: 1)
I0529 13:31:01.608786  266232 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0529 13:31:01.608789  266232 ev_poll_posix.cc:593] FD from fork parent still in poll li

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedded 8 chunks in 1580ms
Index shape: (8, 384)  (8 chunks × 384 dims)
Index memory: 12.0 KB


## 4. Build the Query (Retrieval) Pipeline

In [7]:
def retrieve(query: str, top_k: int = 3) -> list:
    """Embed query and return top_k most similar chunks."""
    q_emb  = embed_model.encode([query])[0]
    norms  = np.linalg.norm(chunk_embeddings, axis=1, keepdims=True)
    normed = chunk_embeddings / (norms + 1e-9)
    q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-9)
    sims   = normed @ q_norm
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(all_chunks[i], float(sims[i])) for i in top_idx]


# ── Test retrieval ────────────────────────────────────────────────────────
test_queries = [
    'How do I diagnose a Stargate crash?',
    'My replication is lagging behind the RPO target',
    'Network packet loss on storage interface',
]

for q in test_queries:
    print(f'Query: "{q}"')
    results = retrieve(q, top_k=2)
    for chunk, score in results:
        print(f'  [{score:.3f}] {chunk["source"]} — {chunk["text"][:80]}...')
    print()

Query: "How do I diagnose a Stargate crash?"
  [0.531] KB-2001 — Stargate is the Nutanix I/O manager responsible for all read and write operation...
  [0.286] KB-2007 — NCC (Nutanix Cluster Check) is the automated health check framework. Run a full ...

Query: "My replication is lagging behind the RPO target"
  [0.554] KB-2002 — Cerebro is the Nutanix data protection service managing async replication, NearS...
  [0.266] KB-2007 — NCC (Nutanix Cluster Check) is the automated health check framework. Run a full ...

Query: "Network packet loss on storage interface"
  [0.360] KB-2006 — MTU mismatch between CVMs, OVS bridges, and physical switches causes silent pack...
  [0.165] KB-2007 — NCC (Nutanix Cluster Check) is the automated health check framework. Run a full ...



## 5. Augmented Generation — RAG Answer

Now we combine retrieval with generation: the retrieved chunks become the context that grounds Gemini's answer in your private knowledge base.

In [8]:
RAG_SYSTEM = """You are a Nutanix support assistant.
Answer questions using ONLY the provided context passages.
If the answer is not in the provided context, respond with: "I don't have that information in my knowledge base."
Always cite the KB source (e.g. "[KB-2001]") for every claim you make."""

def rag_answer(question: str, top_k: int = 3) -> dict:
    """Full RAG pipeline: retrieve → augment → generate."""
    # 1. Retrieve
    retrieved = retrieve(question, top_k=top_k)

    # 2. Build context block
    context_parts = []
    for chunk, score in retrieved:
        context_parts.append(
            f"[{chunk['source']} — {chunk['title']}]\n{chunk['text']}"
        )
    context = '\n\n'.join(context_parts)

    # 3. Generate
    user_msg = f"Context:\n{context}\n\nQuestion: {question}"
    m = genai.GenerativeModel(MODEL, system_instruction=RAG_SYSTEM)
    resp = m.generate_content(
        user_msg,
        generation_config=genai.GenerationConfig(max_output_tokens=350, temperature=0.1)
    )

    return {
        'question':  question,
        'answer':    resp.text,
        'retrieved': retrieved,
    }


# ── Test RAG answer ───────────────────────────────────────────────────────
result = rag_answer('What exact command do I use to check Stargate status?')

print(f"Q: {result['question']}")
print(f"\nRetrieved chunks:")
for chunk, score in result['retrieved']:
    print(f"  [{score:.3f}] {chunk['source']}")
print(f"\nRAG Answer:")
print(result['answer'])

Q: What exact command do I use to check Stargate status?

Retrieved chunks:
  [0.392] KB-2001
  [0.338] KB-2007
  [0.172] KB-2007

RAG Answer:
To check the status of the Stargate service, use the following command: `genesis status | grep -i stargate` [KB-2001].


## 6. RAG vs No-RAG — Hallucination Comparison

We run the same questions through both paths and observe where direct LLM answers are vague, wrong, or invented, and where RAG provides specific, grounded answers.

> **Instructor Note:** Run this cell and ask attendees to spot the differences. Key things to look for: RAG answers cite KB source IDs, contain specific CLI commands, and stay within the scope of the actual KB content. Direct answers may be correct in general but often miss Nutanix-specific details or confidently add incorrect ones.

In [9]:
import pandas as pd

comparison_questions = [
    'What CLI command clears Stargate WAL corruption?',
    'How do I check if NTP clock skew is causing replication lag?',
    'Where are the Prism Central replica logs located?',   # not in our KB
    'What LACP timer mode does Nutanix use?',
    'How do I run a full NCC health check?',
]

print('=' * 90)
print(f'{"Question":<42}  {"Direct (no context)":<22}  {"RAG answer (grounded)"}')
print('=' * 90)

results = []
for q in comparison_questions:
    direct = ask_direct(q, max_tokens=120).strip().replace('\n', ' ')[:110]
    rag    = rag_answer(q, top_k=3)['answer'].strip().replace('\n', ' ')[:110]
    results.append({'question': q, 'direct': direct, 'rag': rag})
    print(f'\nQ: {q}')
    print(f'  Direct : {direct}')
    print(f'  RAG    : {rag}')

print()
print('Observations:')
print('  ✅ RAG answers cite specific KB sources and exact CLI commands')
print('  ✅ RAG refuses to answer when the KB lacks the information')
print('  ⚠️  Direct answers may be plausible but could be hallucinated details')

Question                                    Direct (no context)     RAG answer (grounded)

Q: What CLI command clears Stargate WAL corruption?
  Direct : To clear Stargate Write-Ahead Log (WAL) corruption, you typically need to remove the corrupted segment files f
  RAG    : To clear Stargate WAL corruption, you must stop Stargate with `genesis stop_tasks`, rename the WAL directory t

Q: How do I check if NTP clock skew is causing replication lag?
  Direct : To determine if NTP clock skew is causing replication lag, you need to distinguish between **clock drift** (th
  RAG    : To check if NTP clock skew is causing replication lag, run the command `ntpq -p` on both sites. The offset sho

Q: Where are the Prism Central replica logs located?
  Direct : In Nutanix Prism Central (PC), the logs related to replication, database synchronization, and the underlying s
  RAG    : I don't have that information in my knowledge base.

Q: What LACP timer mode does Nutanix use?
  Direct : Nutanix AHV

## 7. Lab Summary

| Component | What it does | Tool used |
|-----------|-------------|-----------|
| **Chunker** | Splits documents into ~120-word overlapping passages | Python |
| **Embedder** | Converts text to 384-dim semantic vectors | `sentence-transformers` |
| **Index** | Stores embeddings + metadata in memory | `numpy` array |
| **Retriever** | Cosine similarity → top-k matching chunks | `numpy` |
| **Generator** | Grounds LLM answer in retrieved context | Gemini API |

### What RAG does NOT solve
- **Retrieval miss**: if the answer is not in the KB, RAG correctly says "I don't know"
- **Context window overflow**: very long retrievals still exceed LLM input limits
- **Stale knowledge**: retrieved documents may be outdated (need scheduled re-ingestion)

> **Instructor Note:** Lab 5.5 scales this pattern to 7 full KB articles, wraps it in a clean class, and adds systematic evaluation of retrieval quality and hallucination reduction.

---
## 🎯 Challenges

### Challenge 1 — Chunk Size Impact
Re-index the documents with `chunk_size=60` and `chunk_size=250`. For the same test questions, compare which chunk size gives better retrieval scores. What is the trade-off?

### Challenge 2 — Re-ranking
After retrieving `top_k=6` chunks, implement a simple re-ranker: ask Gemini to score each chunk 1–5 for relevance to the question, then keep only the top 3 highest-scored. Does this improve answer quality?

### Challenge 3 — Source Attribution
Modify `rag_answer()` to append a **Sources** section at the end of every answer listing the KB articles used, formatted as a bulleted list. Ensure Gemini only lists sources it actually cited in the answer.

In [ ]:
# Challenge workspace
